In [ ]:
from google.colab import drive
import shutil
import os
import json

# Mount Drive
drive.mount('/content/drive')

# Copy images to local SSD (faster than reading from Drive)
drive_path = '/content/drive/MyDrive/CMPE246/Dataset'
local_path = '/content/waste_data'

if not os.path.exists(local_path):
    print("Copying images...")
    shutil.copytree(drive_path, local_path)
    print("Done.")

# Show what we have
print("\n--- Dataset ---")
total = 0
for cls in sorted(os.listdir(local_path)):
    cls_path = os.path.join(local_path, cls)
    if os.path.isdir(cls_path):
        count = len(os.listdir(cls_path))
        total += count
        print(f"  {cls}: {count} images")
print(f"  TOTAL: {total} images")

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Augmentation to help with glossy paper / lighting issues
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=False,
    brightness_range=[0.6, 1.4],
    channel_shift_range=30.0,
    fill_mode='nearest'
)

# No augmentation on validation
val_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_generator = train_datagen.flow_from_directory(
    '/content/waste_data',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_generator = val_datagen.flow_from_directory(
    '/content/waste_data',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

NUM_CLASSES = train_generator.num_classes
class_names = list(train_generator.class_indices.keys())
print(f"Classes ({NUM_CLASSES}): {class_names}")

In [ ]:
# Build or load model
import glob as g
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model

model = None

# Check if theres a saved model we can use
search_paths = [
    '/content/drive/MyDrive/CMPE246/waste_classifier_*.keras',
    '/content/drive/MyDrive/waste_classifier_*.keras',
    '/content/drive/MyDrive/CMPE246/waste_classifier_*.h5',
    '/content/drive/MyDrive/waste_classifier_*.h5',
]

found_models = []
for pattern in search_paths:
    found_models.extend(g.glob(pattern))

if found_models:
    model_path = max(found_models, key=os.path.getmtime)
    print(f"Found model: {model_path}")
    try:
        from tensorflow.keras.models import load_model
        model = load_model(model_path)
        print("Model loaded successfully.")

        # If classes changed, swap the output layer
        if model.output_shape[-1] != NUM_CLASSES:
            print(f"Class count changed ({model.output_shape[-1]} -> {NUM_CLASSES}). Rebuilding head...")
            base_output = model.layers[-3].output
            x = Dropout(0.3)(base_output)
            predictions = Dense(NUM_CLASSES, activation='softmax')(x)
            model = tf.keras.Model(inputs=model.input, outputs=predictions)
            print("New head attached.")
    except Exception as e:
        print(f"Could not load model: {e}")
        print("Building fresh instead...\n")
        model = None

if model is None:
    # Build new MobileNetV2
    base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    base.trainable = False  # freeze base

    x = base.output
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.3)(x)
    predictions = Dense(NUM_CLASSES, activation='softmax')(x)
    model = Model(inputs=base.input, outputs=predictions)
    print("Built new MobileNetV2 model.")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# Training
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)

VERSION = "V6"
save_path = f'/content/drive/MyDrive/waste_classifier_{VERSION}.keras'

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint(save_path, monitor='val_loss', save_best_only=True, verbose=1),
]

print(f"Training {VERSION}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=30,
    callbacks=callbacks,
    verbose=1
)

# Save class map
indices_path = f'/content/drive/MyDrive/class_indices_{VERSION}.json'
with open(indices_path, 'w') as f:
    json.dump(train_generator.class_indices, f, indent=2)

print(f"\nModel saved to: {save_path}")
print(f"Class map saved to: {indices_path}")

In [ ]:
# Training curves + confusion matrix
# (run after Cell 4, if runtime restarted re-run Cells 1-4 first)
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

try:
    history
except NameError:
    raise RuntimeError("No training history found. Run Cell 4 first.")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['accuracy'], label='Train')
ax1.plot(history.history['val_accuracy'], label='Val')
ax1.set_title('Accuracy')
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(True)

ax2.plot(history.history['loss'], label='Train')
ax2.plot(history.history['val_loss'], label='Val')
ax2.set_title('Loss')
ax2.set_xlabel('Epoch')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

# Confusion matrix
val_generator.reset()
y_pred_probs = model.predict(val_generator, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = val_generator.classes

print("\n--- Classification Report ---")
print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(8, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, cmap='Blues', values_format='d')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Re-sync dataset from Drive (run this if you added new images)
# Then re-run Cells 2, 3, 4, 5

if os.path.exists(local_path):
    shutil.rmtree(local_path)
    print("Cleared old cache.")

print("Re-syncing from Drive...")
shutil.copytree(drive_path, local_path)

print("\n--- Updated Dataset ---")
total = 0
for cls in sorted(os.listdir(local_path)):
    cls_path = os.path.join(local_path, cls)
    if os.path.isdir(cls_path):
        count = len(os.listdir(cls_path))
        total += count
        print(f"  {cls}: {count} images")
print(f"  TOTAL: {total} images")
print("\nNow re-run Cells 2 -> 3 -> 4 -> 5 to retrain.")

In [ ]:
# Export TFLite model for Pi
import tensorflow as tf
import numpy as np
import json

def representative_dataset():
    """Sample images for INT8 calibration."""
    val_generator.reset()
    for i in range(min(100, len(val_generator))):
        batch, _ = val_generator[i]
        for img in batch:
            yield [np.expand_dims(img.astype(np.float32), axis=0)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.float32
converter.inference_output_type = tf.float32

tflite_model = converter.convert()

# Save
tflite_path = f'/content/drive/MyDrive/CMPE246/waste_classifier_{VERSION}.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

class_map_path = f'/content/drive/MyDrive/CMPE246/class_indices_{VERSION}.json'
with open(class_map_path, 'w') as f:
    json.dump(train_generator.class_indices, f, indent=2)

size_mb = len(tflite_model) / (1024 * 1024)
print(f"TFLite model: {tflite_path} ({size_mb:.1f} MB)")
print(f"Class map: {class_map_path}")
print(f"\nCopy both files to the Pi with scp")

In [ ]:
# Live camera test (browser webcam)
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import numpy as np
import cv2
import io
from PIL import Image as PILImage

# Load class names
with open(f'/content/drive/MyDrive/class_indices_{VERSION}.json', 'r') as f:
    idx_map = json.load(f)
class_names_live = {v: k for k, v in idx_map.items()}

CONFIDENCE_THRESHOLD = 70.0

# Which classes go to recycling vs garbage
BIN_MAP = {
    'glass':   'RECYCLING',
    'metal':   'RECYCLING',
    'paper':   'RECYCLING',
    'plastic': 'RECYCLING',
    'other':   'GARBAGE',
}

def video_stream():
    js = Javascript('''
    var video, div = null, stream, captureCanvas, labelElement, binElement;

    function removeDom() {
        stream.getTracks().forEach(track => track.stop());
        video.remove(); div.remove(); video = null; div = null;
    }

    async function createDom() {
        if (div !== null) return;
        div = document.createElement('div');
        div.style.cssText = 'border:2px solid #333; padding:10px; width:max-content; font-family:monospace;';

        video = document.createElement('video');
        video.style.display = 'block';
        video.width = 640; video.height = 480; video.autoplay = true;
        stream = await navigator.mediaDevices.getUserMedia({video: true});
        video.srcObject = stream;

        labelElement = document.createElement('div');
        labelElement.style.cssText = 'font-size:24px; font-weight:bold; margin-top:10px;';
        labelElement.innerText = 'Starting...';

        binElement = document.createElement('div');
        binElement.style.cssText = 'font-size:20px; margin-top:5px; padding:5px;';

        document.body.appendChild(div);
        div.appendChild(video);
        div.appendChild(labelElement);
        div.appendChild(binElement);

        captureCanvas = document.createElement('canvas');
        captureCanvas.width = 640; captureCanvas.height = 480;
    }

    async function takePhoto() {
        await createDom();
        captureCanvas.getContext('2d').drawImage(video, 0, 0, 640, 480);
        return captureCanvas.toDataURL('image/jpeg', 0.8);
    }

    window.setLabel = function(text, color) {
        labelElement.innerText = text;
        labelElement.style.color = color;
    }
    window.setBin = function(text, color) {
        binElement.innerText = text;
        binElement.style.color = color;
    }
    ''')
    display(js)

def preprocess_frame(data):
    binary = b64decode(data.split(',')[1])
    img = PILImage.open(io.BytesIO(binary)).convert('RGB')
    img = img.resize((224, 224))
    return np.expand_dims(np.array(img) / 255.0, axis=0)

video_stream()
print("Live detection running... (stop cell to end)")

while True:
    data = eval_js('takePhoto()')
    img_array = preprocess_frame(data)

    preds = model.predict(img_array, verbose=0)
    idx = np.argmax(preds)
    conf = np.max(preds) * 100
    label = class_names_live[idx]

    if conf < CONFIDENCE_THRESHOLD:
        eval_js('setLabel("UNCERTAIN - move item closer", "#FF8C00")')
        eval_js('setBin("Waiting...", "#888")')
    else:
        bin_label = BIN_MAP.get(label, 'GARBAGE')
        color = '#2E8B57' if bin_label == 'RECYCLING' else '#DC143C'
        eval_js(f'setLabel("{label.upper()} ({conf:.1f}%)", "{color}")')
        eval_js(f'setBin("-> Open {bin_label} bin", "{color}")')